# Human vs. Monkey — Training

Place class folders directly inside `data/human-vs-monkey/`, or change `dataset_dir`.

In [ ]:
import glob
import cv2 as cv
import numpy as np
from keras.utils import to_categorical
from keras.models import Model,save_model
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from keras.layers import Input,Conv2D,MaxPooling2D,Dense,ReLU,Add,GlobalAveragePooling2D,BatchNormalization

In [ ]:
data=[]
labels=[]

In [ ]:
from pathlib import Path

dataset_dir = Path("data/human-vs-monkey")

In [ ]:
for item in sorted(dataset_dir.glob("*/*")):
    img = cv.imread(str(item)) if item.is_file() else None
    if img is None:
        continue
    f_img = cv.bilateralFilter(img , d=9 , sigmaColor=75 , sigmaSpace=75)
    std_dev = np.std(f_img)
    print(std_dev)
    if std_dev>=0 and std_dev <= 50 :
        alpha = 1.5
    elif std_dev>50 and std_dev <= 100 :
        alpha = 1.0
    elif std_dev > 100 and std_dev <=150 :
        alpha = 0.75
    elif std_dev > 150 :
        alpha = 0.50
    reso_img = cv.convertScaleAbs(f_img,alpha=alpha,beta=0)
    lab = cv.cvtColor(reso_img,cv.COLOR_RGB2Lab)
    l,a,b = cv.split(lab)
    image = cv.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
    l=image.apply(l)
    lab=cv.merge((l,a,b))
    n_img = cv.cvtColor(lab,cv.COLOR_Lab2RGB)
    r_img = cv.resize(n_img,(25 , 25))
    img = cv.normalize(r_img,None,0,255,cv.NORM_MINMAX)
    data.append(img)
    label = item.parent.name
    labels.append(label)

In [ ]:
data =np.array(data)
le=LabelEncoder()
labels=le.fit_transform(labels)
labels=to_categorical(labels)

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(data,labels,test_size=0.1)

In [ ]:
inputs = Input(shape=(25,25,3))

x=Conv2D(128,(7,7),strides=2,padding="same")(inputs)
x=BatchNormalization()(x)
x=ReLU()(x)

x=MaxPooling2D((3,3),strides=2,padding="same")(x)

for filters,blocks,stride in zip([16,32,64,128],[3,5,3,5],[2,4,3,2]):
    for blocks in range(blocks):
        short=x
        if blocks==0:
            short=Conv2D(4*filters,(1,1),strides=stride,padding="same")(x)
            short=BatchNormalization()(short)
            x=Conv2D(filters,(1,1),strides=stride,padding="same")(x)
        else:
            x=Conv2D(filters,(1,1),strides=1,padding="same")(x)

        x=BatchNormalization()(x)
        x=ReLU()(x)

        x=Conv2D(256,(3,3),strides=1,padding="same")(x)
        x=BatchNormalization()(x)
        x=ReLU()(x)

        x=Conv2D(4*filters,(1,1),strides=1,padding="same")(x)
        x=BatchNormalization()(x)

        x=Add()([short,x])

x=GlobalAveragePooling2D()(x)
out_puts=Dense(2,activation="softmax")(x)
net=Model(inputs,out_puts)

In [ ]:
net.compile(optimizer="adamax",loss="categorical_crossentropy",metrics=["accuracy"])

In [ ]:
net.fit(x_train,y_train,batch_size=100,epochs=1,validation_data=(x_test,y_test))

In [ ]:
net.save("human-Or-monkey_MODEL.keras")